In [1]:
from pyspark.sql import SparkSession as ss

spark = ss.builder    .appName('calc')     .master("local[*]")     .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")     .config("spark.hadoop.fs.s3a.endpoint", "http://minio-storage:9000")     .config("spark.hadoop.fs.s3a.access.key", "minioadmin")     .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")     .config("spark.hadoop.fs.s3a.path.style.access", "true")     .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")     .getOrCreate()


:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-49752259-db06-460b-8546-c559b4111c15;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (113ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (12817ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7.

In [2]:
# 1. 이벤트 퍼널
spark.read.parquet("s3a://test-bucket/gold/event_funnel").show()


26/07/31 08:00:33 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+----------+--------+
|event_type|   count|
+----------+--------+
|  purchase|  660281|
|      view|39459640|
|      cart| 2118952|
+----------+--------+



In [3]:
# 2. 카테고리별 매출 Top 10
spark.read.parquet("s3a://test-bucket/gold/category_revenue").orderBy("revenue", ascending=False).show(10, truncate=False)


+--------------------------------+--------------------+--------------+
|category_code                   |revenue             |purchase_count|
+--------------------------------+--------------------+--------------+
|electronics.smartphone          |1.7780273615995994E8|382630        |
|electronics.video.tv            |1.2455886240000386E7|30268         |
|computers.notebook              |1.065858183999996E7 |18415         |
|electronics.clocks              |6266850.410000084   |21458         |
|appliances.kitchen.washer       |5788619.26000018    |19687         |
|electronics.audio.headphone     |5666697.000000083   |40749         |
|appliances.kitchen.refrigerators|4091056.2599999523  |10429         |
|appliances.environment.vacuum   |2757957.5899999226  |18125         |
|computers.desktop               |1532778.3599999985  |3574          |
|electronics.tablet              |1519396.5099999977  |6123          |
+--------------------------------+--------------------+--------------+
only s

In [4]:
# 브랜드별 매출 Top 10
spark.read.parquet("s3a://test-bucket/gold/brand_revenue").orderBy("revenue", ascending=False).show(10, truncate=False)


+-------+--------------------+--------------+
|brand  |revenue             |purchase_count|
+-------+--------------------+--------------+
|apple  |1.274953047600078E8 |165693        |
|samsung|5.4794830150003776E7|198674        |
|xiaomi |1.0906490590000033E7|58038         |
|lg     |5031784.449999997   |11833         |
|huawei |4786750.450000096   |23489         |
|oppo   |3488540.759999957   |15080         |
|acer   |3355824.6899999646  |6424          |
|lenovo |2714546.9700000277  |6591          |
|asus   |1672505.559999989   |3026          |
|hp     |1340006.0700000073  |4013          |
+-------+--------------------+--------------+
only showing top 10 rows



In [ ]:
# 3. 일별 이벤트 추이
spark.read.parquet("s3a://test-bucket/gold/daily_events").orderBy("event_date", "event_type").show(30, truncate=False)


In [ ]:
# 일별 구매 건수/매출
spark.read.parquet("s3a://test-bucket/gold/daily_purchase").orderBy("event_date").show(30, truncate=False)


In [ ]:
# 4. 가격 기술통계 (전체 / purchase만)
print("=== 전체 price ===")
spark.read.parquet("s3a://test-bucket/gold/price_stats").show()

print("=== 백분위수 ===")
spark.read.parquet("s3a://test-bucket/gold/price_percentiles").show()

print("=== purchase이벤트만의 price ===")
spark.read.parquet("s3a://test-bucket/gold/purchase_price_stats").show()
